# Deploy a Model

Deploy a model via the **RHOAI Dashboard** (Models as a Service), then verify and save the config.

**Prerequisites:**
- `1_environment_setup.ipynb` completed (cluster verified, `.env` configured)
- MaaS infrastructure installed via [RHOAI-Toolkit](https://github.com/hyogrin/RHOAI-Toolkit)

## 1. Deploy Model via RHOAI Dashboard

### Recommended Models

| Model | VRAM | Source (OCI ModelCar) |
|-------|------|----------------------|
| Qwen3.5-35B-A3B MoE FP8 | ~21 GB | `oci://registry.redhat.io/rhai/modelcar-qwen3-5-35b-a3b-fp8-dynamic:3.0` |
| Qwen3-14B | ~14 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen3-14b` |
| Qwen3-8B | ~8 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen3-8b` |
| Qwen3-4B | ~4 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen3-4b` |
| Qwen2.5-7B-Instruct | ~7 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen2.5-7b-instruct` |

### Steps

1. Open **RHOAI Dashboard** → **AI Hub** → **Models** → **Deployments** → **Deploy**
2. Select **Distributed inference with llm-d**
3. Fill in:
   - **Model name**: e.g., `qwen35-35b`
   - **Project**: `demo` (create via Data Science Projects if needed)
   - **Model source**: Select **OCI** and enter the URI from the table above
   - **Hardware profile**: Select a GPU profile (enable in Settings → Hardware profiles if disabled)
4. Expand **Custom parameters** and add the values from the table below
5. Click **Deploy** and wait for Ready (5-15 min for large models)

### Custom Parameters (vLLM)

Add these in the Dashboard's **Custom parameters** section as environment variables:

| Key | Value | Why |
|-----|-------|-----|
| `VLLM_ADDITIONAL_ARGS` | See below | vLLM CLI flags |

**Recommended `VLLM_ADDITIONAL_ARGS` value:**

```
--max-model-len=16384 --enforce-eager --gpu-memory-utilization=0.90 --enable-auto-tool-choice
 --tool-call-parser=hermes --chat-template=/etc/chat-template/chat_template.jinja --reasoning-parser=qwen3


```

| Flag                                                     | Effect                                                                                                                                                                                                                                                                 |
| -------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `--max-model-len=16384`                                  | Caps the **combined prompt and output length to 16K tokens**. Reducing it from the model’s native context length lowers KV-cache memory requirements and helps prevent OOM.                                                                                            |
| `--enforce-eager`                                        | Disables CUDA graphs and always uses PyTorch eager execution. This can reduce CUDA-graph capture memory and startup overhead, but may lower inference throughput. It is **not always required** for large models.                                                      |
| `--gpu-memory-utilization=0.95`                          | Allows the vLLM model executor to use up to **95% of each GPU’s VRAM**. Remaining memory after model weights and runtime allocations is generally used for KV cache. Default is `0.9`.                                                                                 |
| `--enable-auto-tool-choice`                              | Enables the model to automatically decide whether to return a normal response or invoke one or more tools.                                                                                                                                                             |
| `--tool-call-parser=qwen3_coder`                         | Parses the **Qwen3-Coder-specific tool-call format**, used by models such as `Qwen3-Coder-30B-A3B-Instruct` and `Qwen3-Coder-480B-A35B-Instruct`.                                                                                                                      |
| `--tool-call-parser=hermes`                              | Parses the **Hermes-style `<tool_call>` format**, used by Qwen2.5, QwQ and Hermes-compatible models.                                                                                                                                                                   |
| `--trust-remote-code`                                    | Allows Hugging Face model repositories to load and execute their custom Python model or tokenizer code. Use only with trusted model repositories.                                                                                                                      |
| `--reasoning-parser=qwen3`                               | Separates Qwen3 reasoning output from the final answer and returns it in the OpenAI-compatible `reasoning_content` field. you can turn off to create </think> tag for the coding assistant use case                                                                                                                                              |
| `--chat-template=/etc/chat-template/chat_template.jinja` | Uses the specified Jinja2 chat template to convert OpenAI-format messages, roles and tool definitions into the model-specific prompt format. It is useful when the tokenizer does not provide the correct template or when a custom tool-calling template is required. |







> **Without `--max-model-len` and `--enforce-eager`**, large models (27B+) will likely fail to start on a single GPU due to OOM during KV cache allocation.

## 2. Verify Deployment

After deploying from the Dashboard, verify the model is running.

In [4]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")
NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")

print(f"=== LLMInferenceService in '{NAMESPACE}' ===")
r = subprocess.run(["oc", "get", "llminferenceservice", "-n", NAMESPACE], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 and r.stdout.strip() else "(none)")

print(f"\n=== Pods in '{NAMESPACE}' ===")
r = subprocess.run(["oc", "get", "pods", "-n", NAMESPACE], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "(no pods)")

print("\nTip: Re-run this cell until READY=True, then proceed to Step 3.")

=== LLMInferenceService in 'demo' ===
(none)

=== Pods in 'demo' ===
(no pods)

Tip: Re-run this cell until READY=True, then proceed to Step 3.


## 3. Save Model Config to `.env`

Persists model info to `../.env` for use in subsequent notebooks.

In [5]:
import subprocess, json, os, re
from dotenv import load_dotenv

load_dotenv("../.env")
NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")

r = subprocess.run(
    ["oc", "get", "llminferenceservice", "-n", NAMESPACE, "-o", "json"],
    capture_output=True, text=True
)

MODEL_NAME = MODEL_URL = None
if r.returncode == 0:
    items = json.loads(r.stdout).get("items", [])
    for item in items:
        conditions = item.get("status", {}).get("conditions", [])
        ready = next((c["status"] for c in conditions if c["type"] == "Ready"), "False")
        if ready == "True":
            MODEL_NAME = item["metadata"]["name"]
            MODEL_URL = item["status"].get("url", "")
            break

if not MODEL_NAME:
    print("⚠️  No Ready model found. Wait for deployment to complete and re-run.")
else:
    CLUSTER_DOMAIN = subprocess.run(
        ["oc", "get", "ingresses.config.openshift.io", "cluster", "-o", "jsonpath={.spec.domain}"],
        capture_output=True, text=True
    ).stdout.strip()

    env_path = os.path.abspath("../.env")
    with open(env_path) as f:
        content = f.read()

    updates = {
        "CLUSTER_DOMAIN": CLUSTER_DOMAIN,
        "MODEL_NAME": MODEL_NAME,
        "MODEL_NAMESPACE": NAMESPACE,
        "MODEL_ENDPOINT": MODEL_URL,
    }

    for key, value in updates.items():
        if not value:
            continue
        pattern = rf"^#?\s*{key}=.*$"
        replacement = f"{key}={value}"
        if re.search(pattern, content, re.MULTILINE):
            content = re.sub(pattern, replacement, content, count=1, flags=re.MULTILINE)
        else:
            content = content.rstrip("\n") + f"\n{replacement}\n"

    with open(env_path, "w") as f:
        f.write(content)

    print(f"✅ Saved to {env_path}:")
    for k, v in updates.items():
        print(f"   {k}={v}")

⚠️  No Ready model found. Wait for deployment to complete and re-run.


## 4. Test Model Endpoint

Quick inference test via `port-forward` to bypass gateway ext_proc.

In [6]:
import subprocess, json, time, os
from dotenv import load_dotenv

load_dotenv("../.env", override=True)
MODEL_NAME = os.getenv("MODEL_NAME")
NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")

if not MODEL_NAME:
    print("⚠️  MODEL_NAME not set. Run Step 3 first.")
else:
    svc_name = f"{MODEL_NAME}-kserve-workload-svc"
    local_port = 18000

    pf = subprocess.Popen(
        ["oc", "port-forward", f"svc/{svc_name}", f"{local_port}:8000", "-n", NAMESPACE],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(3)

    try:
        r = subprocess.run(
            ["curl", "-sk", "--max-time", "30",
             f"https://localhost:{local_port}/v1/chat/completions",
             "-H", "Content-Type: application/json",
             "-d", json.dumps({
                 "model": MODEL_NAME,
                 "messages": [{"role": "user", "content": "Write a Python hello world one-liner."}],
                 "max_tokens": 30
             })],
            capture_output=True, text=True, timeout=35
        )
        if r.returncode == 0 and r.stdout.strip():
            resp = json.loads(r.stdout)
            if "choices" in resp:
                print(f"✅ Response:\n{resp['choices'][0]['message']['content']}")
            elif "error" in resp:
                print(f"❌ API error: {resp['error']}")
            else:
                print(f"❌ Unexpected: {r.stdout[:300]}")
        else:
            print(f"❌ No response. Check: oc get pods -n {NAMESPACE}")
    finally:
        pf.terminate()
        pf.wait()

❌ No response. Check: oc get pods -n demo


## Next Steps

- `3_app_setup.ipynb` — Build and deploy the cafe-order-system demo app
- `../1_mcp_servers/2_deploy_mcp_servers.ipynb` — Verify MCP tool servers
- `../2_maas/2_enable_maas.ipynb` — Register models and create API keys